In [31]:
# Model Inference Cell - Run this cell for predictions
import joblib
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [32]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class StructuredFeaturesTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Convert features column to numpy array
        return np.array(X.tolist(), dtype=np.float32)

# Custom transformer for text data
class TextTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Join stemmed text back into strings
        return X.apply(lambda x: ' '.join(x))

In [33]:
# ============================================================================
# 1. LOAD THE TRAINED MODEL
# ============================================================================
print("🔄 Loading trained model...")
try:
    # Load the best pipeline
    best_pipeline = joblib.load('best_lgbm_pipeline.joblib')
    
    # Load best parameters (optional)
    try:
        best_params = joblib.load('best_parameters.joblib')
        print("✅ Model and parameters loaded successfully!")
    except:
        print("✅ Model loaded successfully! (Parameters file not found)")
        best_params = None
    
    # Display model info
    print(f"📊 Model type: {type(best_pipeline.named_steps['model']).__name__}")
    print(f"🔧 Pipeline steps: {list(best_pipeline.named_steps.keys())}")
    
except FileNotFoundError:
    print("❌ Error: Model file 'best_lgbm_pipeline.joblib' not found!")
    print("   Please ensure you've run the training script and saved the model.")
    raise


🔄 Loading trained model...
✅ Model and parameters loaded successfully!
📊 Model type: LGBMClassifier
🔧 Pipeline steps: ['preprocessing', 'model']
✅ Model and parameters loaded successfully!
📊 Model type: LGBMClassifier
🔧 Pipeline steps: ['preprocessing', 'model']


In [34]:
import re
from nltk.tokenize import word_tokenize
def preprocess_text(text):
    # remove non-alphabet characters
    text = re.sub(r'[^A-Za-z0-9!?]', ' ', text)
    text = re.sub(r'([!?])', r' \1 ', text)
    # remove whitespace
    text = text.strip()
    # remove newline
    text = text.replace('\n', ' ')
    # remove extra space
    text = re.sub(' +', ' ', text)
    # lowercase
    text = text.lower()
    # tokenize
    text = word_tokenize(text)
    # replace punctuation tokens
    text = ['EXCLAMATIONTOKEN' if token == '!' else 'QUESTIONTOKEN' if token == '?' else token for token in text]
    result = text
    return result


In [35]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
 
factory = StemmerFactory()
stopword_factory = StopWordRemoverFactory()
stopword_indonesia = stopword_factory.get_stop_words()
stemmer = factory.create_stemmer()

In [36]:
angka = ['nol', 'satu', 'dua', 'tiga', 'empat', 'lima', 'enam', 'tujuh', 'delapan', 'sembilan']
potential_clickbait_words = ['bikin', 'viral', 'gara', 'fakta', 'kejut', 'kamu', 'wajib', 'pakai', 'ala', 'heboh', 'geger', 'video', 'foto', 'cantik']

def count_full_word_capitals(text):
    return sum(len(word) for word in text.split() if word.isupper() and len(word) > 1)

def count_all_caps_words(text):
    return sum(1 for word in text.split() if word.isupper() and len(word) > 1)

def check_features(text):
    words = text.split()
    word_count = len(words)
    avg_word_length = sum(len(w) for w in words) / word_count if word_count > 0 else 0

    stemmed_words = [stemmer.stem(word.lower()) for word in words]

    result = []

   # Feature 1: contains exclamation mark
    result.append(1 if '!' in text else 0)
    # Feature 2: contains question mark
    result.append(1 if '?' in text else 0)
    result.append(1 if '??' in text else 0)
    # Feature 3: contains multiple exclamation marks
    result.append(1 if '!!' in text else 0)
    # Feature 4: contains multiple dots (ellipsis indicator)
    result.append(1 if '..' in text else 0)
    # Feature 5: contains potential clickbait words (after stemming)
    result.append(1 if any(stem in potential_clickbait_words for stem in stemmed_words) else 0)
    result.append(sum(1 for stem in stemmed_words if stem in potential_clickbait_words))
    # Feature 6: count of exclamation marks
    result.append(text.count('!'))
    # Feature 7: total capital letters from full caps words
    result.append(count_full_word_capitals(text))
    # Feature 8: total length of text (in characters)
    result.append(len(text))    
    # Feature 9: word count
    result.append(word_count)
    # Feature 10: average word length
    result.append(avg_word_length)
    # Feature 11: starts with number (digit or number word in Indonesian)
    result.append(1 if words and (words[0].isdigit() or words[0].lower() in angka) else 0)
    # Feature 12: count of full-uppercase words
    result.append(count_all_caps_words(text))    
    # Feature 13: ratio of uppercase characters to total characters
    result.append(sum(1 for c in text if c.isupper()) / len(text) if text else 0)
    result.append(len(re.findall(r'\d+', text)))
    result.append(int(bool(re.search(r'^\d+', text))))
    result.append(sum(1 for w in words if len(w) <= 2))
    result.append(sum(1 for w in words if len(w) >= 10))

    return result

In [37]:
# POST with text_input as body
text_input = "Viral! Dustin Tiffany Mengatakan Jokowi Koma!"
df = pd.DataFrame([text_input], columns=['title'])
df['preprocessed_text'] = df['title'].apply(preprocess_text)
df['stemmed_text'] = df['preprocessed_text'].apply(lambda x: [stemmer.stem(word.lower()) for word in x])
df['features'] = df['title'].apply(check_features)

prediction = best_pipeline.predict(df[['stemmed_text', 'features']])
prediction_proba = best_pipeline.predict_proba(df[['stemmed_text', 'features']])
# If predict_proba returns a 2D array (n_samples, n_classes), take the first row for this single sample
if hasattr(prediction_proba, 'ndim') and prediction_proba.ndim > 1:
    prediction_proba_row = prediction_proba[0]
else:
    prediction_proba_row = np.array(prediction_proba)

# Debug prints
print(prediction)
print(prediction_proba_row)

classes = best_pipeline.named_steps['model'].classes_
print(classes)
is_clickbait = prediction[0]
label = "clickbait" if is_clickbait == 1 else "non-clickbait"

[1]
[0.0045114 0.9954886]
[0 1]


In [38]:
result = {
    'prediction': label,
    'is_clickbait': is_clickbait,
    'clickbait_probability': {str(cls): float(prob) for cls, prob in zip(classes, prediction_proba_row)},
    'headline': text_input,
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'status': 'success'
}

In [39]:
from pprint import pprint

# print result pretty
pprint(result)

{'clickbait_probability': {'0': 0.004511401684101468, '1': 0.9954885983158985},
 'headline': 'Viral! Dustin Tiffany Mengatakan Jokowi Koma!',
 'is_clickbait': np.int64(1),
 'prediction': 'clickbait',
 'status': 'success',
 'timestamp': '2025-08-24 21:11:36'}
